# Keeks the Geek
A small Unsloth fine-tuning experiment for music, DSP/FFT, physics, AI, programming, creative graphics, drum building and Arduino synthesizers, plus TidalCycles, Hydra, TouchDesigner, Gibber, ChucK, drum sampling and synthesis, live coding and graphics docs.

**This notebook has not been GPU-executed in the initial release.** It trains real weights when you run it. The website currently uses clearly labelled starter-note demo mode.

In Colab choose **Runtime → Change runtime type → GPU**. A T4 16 GB is the initial target, not a verified memory guarantee. No paid compute is required by the code; availability depends on your account.


In [ ]:
import os, pathlib, subprocess, sys
run = lambda *args: subprocess.run(list(args), check=True)
run('nvidia-smi')
project = pathlib.Path('/content/keeks-the-geek')
if not project.exists():
    run('git', 'clone', 'https://github.com/xidist/keeks-the-geek.git', str(project))
os.chdir(project)
if subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Local project changes found. Preserve them before updating the checkout.')
run('git', 'pull', '--ff-only')
print('Source revision:')
run('git', 'rev-parse', 'HEAD')
print('Project:', project)


## Install together in a fresh runtime
The dependency resolver follows Unsloth's package constraints. Training runs in separate Python processes, so notebook imports do not affect Unsloth's patch order. A completed run saves the exact package versions. If installation fails, use the current Unsloth installation guide linked in the README before proceeding.


In [ ]:
run(sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt')


## Review the data and run CPU checks
These are synthetic starter examples, not private conversation history. Review `data/seeds.json` before expanding the corpus. Training has 132 rows; validation has 25 rows. Review the 45 creative-coding examples and their 33 source references in `data/creative_seeds.json` and `data/creative_sources.json`. These are original examples informed by docs, not full scraped manuals. All related numerical variants stay together.


In [ ]:
import json
run(sys.executable, 'scripts/build_dataset.py')
run(sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v')
print(json.dumps(json.loads(pathlib.Path('config.json').read_text()), indent=2))
print(json.loads(pathlib.Path('data/train.jsonl').read_text().splitlines()[0]))


## Record the original model
This creates actual base-model answers to the held-out prompts. No results have been supplied in advance. Existing output files are preserved: choose new output paths if you rerun an experiment.


In [ ]:
run(sys.executable, 'scripts/generate.py', '--output', 'outputs/base_eval.jsonl')


## Two-step GPU smoke test
This is a compatibility check, not the final model. The trainer checks native chat-template alignment, sequence lengths and response-only masks. Stop and fix any failure before continuing.


In [ ]:
run(sys.executable, 'scripts/train.py', '--max-steps', '2', '--output-dir', 'outputs/smoke')


## Full experiment
This starts again from the base model and trains for the epochs in `config.json`. Adapter weights, actual metrics, environment and data hashes are saved under `outputs/full`. To avoid overwriting earlier work, a reused nonempty output directory causes an error. Use a new directory or an explicit checkpoint resume.


In [ ]:
run(sys.executable, 'scripts/train.py', '--output-dir', 'outputs/full')


## Compare real outputs
The numerical checks are narrow scalar tests. Open the blinded CSV and score answers before opening the answer key. A small starter evaluation does not establish broad expertise.


In [ ]:
run(sys.executable, 'scripts/generate.py', '--adapter', 'outputs/full/adapter', '--output', 'outputs/tuned_eval.jsonl')
run(sys.executable, 'scripts/compare.py')
print(pathlib.Path('outputs/comparison/numeric_summary.json').read_text())


## Ask your model


In [ ]:
run(sys.executable, 'scripts/generate.py', '--adapter', 'outputs/full/adapter', '--prompt', 'Give me a weekend project combining drum vibration, FFTs and creative graphics.')


## Download the adapter and evidence
This ZIP includes the trained adapter, configuration, manifests, exact installed packages, generated answers and review sheet. It excludes optimizer checkpoints and base-model downloads. Save it before your Colab runtime disconnects. Follow the README to serve it on a GPU; a Colab session is not permanent website hosting.


In [ ]:
import zipfile
archive = pathlib.Path('/content/Keeks_the_Geek_Trained_Adapter.zip')
selected = [pathlib.Path('outputs/full/adapter'), pathlib.Path('outputs/comparison')]
files_to_save = [pathlib.Path('config.json'), pathlib.Path('data/manifest.json'), pathlib.Path('outputs/full/run_manifest.json'), pathlib.Path('outputs/full/pip-freeze.txt'), pathlib.Path('outputs/base_eval.jsonl'), pathlib.Path('outputs/base_eval.meta.json'), pathlib.Path('outputs/tuned_eval.jsonl'), pathlib.Path('outputs/tuned_eval.meta.json')]
for folder in selected:
    files_to_save.extend(p for p in folder.rglob('*') if p.is_file())
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in files_to_save:
        z.write(path, arcname=str(path))
from google.colab import files
files.download(str(archive))
